# Background

Author: Diane Menuz  
Date: 2026-02-04
Station: Bluff

Goal: Create the qc version of the data


# Set Up

## Parameters

In [ ]:
station = "US-UTD"

date_range = '202305241430_202510221630'
parquet_path = r'M:\Shared drives\UGS_Flux\Data_Processing\final_database_tables' 
file_name = f'{station}_{date_range}_raw.parquet'
micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"

## Libraries

In [ ]:
import sys

sys.path.append(micromet_path)
import micromet
from micromet import validate
from micromet import fix_g_values
from micromet import data_cleaning
from micromet import eddy_plots as ed_plot
from micromet import recalculate_albedo


In [ ]:
sys.path.append(r"C:/Users/dmenuz/Documents/scripts/soil_heat/src/")
from soil_heat import storage_calculations as store
from soil_heat import soil_heat

In [ ]:
import pandas as pd
import numpy as np
import importlib

from prettytable import PrettyTable
from typing import Union

from scipy import stats
import matplotlib.pyplot as plt


from typing import List, Dict, Union

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from plotly.offline import iplot

from pathlib import Path
from datetime import date


In [ ]:
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setFormatter(
    logging.Formatter(
        fmt="%(levelname)s [%(asctime)s] %(name)s – %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
logger.addHandler(ch)

## Database Data

In [ ]:
import requests
headdict = {'Accept-Profile': 'groundwater',  
            'Content-Type': 'application/json'}  

table = 'eddy_events'
resp = requests.get(f"https://ugs-koop-umfdxaxiyq-wm.a.run.app/{table}",  
                    headers=headdict)
event_db = pd.DataFrame(resp.json())
event_db['event_date'] = pd.to_datetime(event_db['event_date'])

table = 'eddy_station_metadata'
resp = requests.get(f"https://ugs-koop-umfdxaxiyq-wm.a.run.app/{table}",  
                    headers=headdict)
metadata = pd.DataFrame(resp.json())

In [ ]:
notes = event_db.loc[(event_db.stationid==station) & (event_db.event_type=='station visit'),
                     ['event_date','event_notes']]

table = PrettyTable()
table.field_names = ["Date", "Note"]
table.sortby = "Date"
table.align["Note"] = "l"

rows_as_tuples = [tuple(x) for x in notes.values] 
table.add_rows(rows_as_tuples) # Use the correct data format

print(table)

In [ ]:
# review program updates
updates = event_db.loc[(event_db.stationid==station) & 
                       (event_db.event_type=='program update'),
                       ['event_date','event_notes']]

table = PrettyTable()
table.field_names = ["Date", "Note"]
table.sortby = "Date"
table.align["Note"] = "l"

rows_as_tuples = [tuple(x) for x in updates.values] 
table.add_rows(rows_as_tuples) # Use the correct data format

print(table)

## Read in Data

In [ ]:
file_path = Path (parquet_path, 'raw', file_name)
raw_dat = pd.read_parquet(file_path)

# Apply Corrections

- Apply Soil and any other calibraton corrections before running data through the "finalize" function.  
- This ensures that the physical limits are applied to the final values.

## Fix Incorrect SG Calibration

In [ ]:
# date from table above when calibration fix was applied to program
calibration_fix_date = '2025-12-05 16:39:00' 

In [ ]:
# SOIL PLATE 2 VaALUES WERE INVERTED FOR A PERIOD OF TIME
flipped_g = raw_dat.copy()
flip_end_date = pd.to_datetime('2025-06-01')
mask = flipped_g.index < flip_end_date
flipped_g.loc[mask, 'G_PLATE_2_1_1'] = flipped_g.loc[mask, 'G_PLATE_2_1_1']*-1


In [ ]:
sg_correct = fix_g_values.correct_vars_by_factor(flipped_g, correction_factor=0.05/0.16, 
                           vars_to_correct=['SG_1_1_1','SG_2_1_1'], 
                           min_correction_date='2010-01-01',
                           max_correction_date=calibration_fix_date)
sg_correct = fix_g_values.calculate_new_g_value(sg_correct, '1')
sg_correct = fix_g_values.calculate_new_g_value(sg_correct, '2')

## Fix Precip Calibration

In [ ]:
# date that precip value fixed in program; site specific!!
precip_program_date = pd.to_datetime('2025-12-02 09:13:00') 
correction_factor = 2.54 # this is likely/hopefully the smae for all sites

In [ ]:
precip_fixed = sg_correct.copy() # copying is key- otherwise if you run this twice, you keep changing the values!
mask = precip_fixed.index<precip_program_date
precip_fixed.loc[mask, 'P_1_1_1'] = precip_fixed.loc[mask, 'P_1_1_1']*correction_factor

In [ ]:
# ed_plot.plotlystuff([precip_fixed, sg_correct],
#                     ['P_1_1_1', 'P_1_1_1'])

# Calculate SoilVue G

- Do this before applying physical limits
- SWC values need to be proportion and not percent! They are converted to percent in the Finalize step

In [ ]:
soilvue = precip_fixed.copy()

In [ ]:
# verify that SWC values are proportions
# will get an assertion error if not (which may just indicate data issues, but 
# inspect data more carefully)
min_swc = min(soilvue['SWC_3_1_1'].min(), soilvue['SWC_3_2_1'].min())
max_swc = max(soilvue['SWC_3_1_1'].max(), soilvue['SWC_3_2_1'].max())
print(f'SWC values between {min_swc} and {max_swc}')
assert min_swc >= 0 and max_swc <= 1, "SWC values are not proportions"


In [ ]:
# calculate soil vue 5 cm heat flux plus heat storage

# calculate soil heat storage component
soilvue['SG_3_1_1'] = store.compute_soil_storage_integrated(
    soilvue,
    col_T5 ="TS_3_1_1",
    col_VWC5 = "SWC_3_1_1")

# calculate soil heat flux at 5 cm down 
soilvue['G_SOILVUE'] = soil_heat.compute_heat_flux_conduction(
    soilvue,
    depth1 = 0.05,
    depth2 = 0.10,
    col_T1 = 'TS_3_1_1',
    col_T2 = 'TS_3_2_1',
    col_theta1='SWC_3_1_1',
    col_theta2='SWC_3_2_1',
    )
# calculate G as storage plus g "plate"
soilvue['G_3_1_1'] = soilvue['SG_3_1_1'] + soilvue['G_SOILVUE'] 

In [ ]:
# # review data if you want
subset1 = soilvue[soilvue.index>pd.to_datetime('2025-07-01')]

ed_plot.plotlystuff([subset1, subset1, subset1], 
                    ['G_3_1_1','SG_3_1_1', 'G_SOILVUE'])
ed_plot.plotlystuff([subset1, subset1, subset1], 
                    ['G_1_1_1','G_2_1_1', 'G_3_1_1'])
ed_plot.plotlystuff([subset1, subset1, subset1], 
                    ['SG_1_1_1','SG_2_1_1', 'SG_3_1_1'])

# Micromet Finalize

- This applies physical limits and runs some other corrections

In [ ]:
reformatter = micromet.Reformatter(
    drop_soil=False, logger=logger,)
finalized_df, report, timestamp_results = reformatter.finalize(soilvue)
finalized_df = finalized_df.replace(-9999, np.nan)

In [ ]:
# review and export report
report_name = file_name.replace('raw.parquet', 'report.csv')
report_path = Path(parquet_path, 'micromet_reports', report_name)
report.to_csv(report_path)
print(f'Report exported to {report_path}')
print('\n')
print('Fields with a lot of missing data')
print(report[report.pct_flagged>=0.5])

In [ ]:
validate.validate_flags(finalized_df)

# Address Common Data Issues

In [ ]:
cleaned_df = finalized_df.copy()

## Drop Precip on Field Days

It really did rain on 2024-08-22 and maybe on 2024-04-16	

In [ ]:
# review rain on station visits
mask = (event_db.stationid==f'{station}') & (event_db.event_type=='station visit') 
visits = event_db[mask]
print('Precip events that occurred on station visits day and could be dropped')
visit_precip = cleaned_df.loc[(cleaned_df.index.floor('D').isin(
    visits.event_date.dt.floor('D'))) & (cleaned_df.P_1_1_1>0),'P_1_1_1']
visit_precip

In [ ]:
# turning those precip events to zero, if you want
drop_precip_date = pd.to_datetime(['2025-07-15 18:00:00'])
cleaned_df.loc[cleaned_df.index.isin(drop_precip_date), 'P_1_1_1'] = 0

## Drop Zeros in G Plates

In [ ]:
mask1 = cleaned_df['G_PLATE_1_1_1']==0
mask2 = cleaned_df['G_PLATE_2_1_1']==0
print(f'{mask1.sum()} zero G plate 1 values and {mask2.sum()} zero G plate 2 values')

cleaned_df.loc[mask1, ['G_PLATE_1_1_1','G_1_1_1']] = np.nan
cleaned_df.loc[mask2, ['G_PLATE_2_1_1','G_2_1_1']] = np.nan

# Address Other Data Issues

In [ ]:
final_dat = cleaned_df.copy()

## Soil

In [ ]:
# # drop bad data from G plate 1
bad_date = pd.to_datetime('2025-03-07 00:00')
mask = final_dat.index>bad_date
final_dat.loc[mask, ['G_PLATE_1_1_1', 'G_1_1_1']] = np.nan

In [ ]:
# # drop bad data from plate 2

bad_start = pd.to_datetime('2025-05-30 6:00')
bad_end = pd.to_datetime('2025-07-18 23:00')
mask = (final_dat.index>bad_start) & (final_dat.index<bad_end)
final_dat.loc[mask, ['G_PLATE_2_1_1', 'G_2_1_1']] = np.nan

In [ ]:
# SOILVUE 
# drop spikes
drop_value = 3.5
bad_soilvue_mask = (final_dat.K_3_7_1<drop_value)
bad_data = final_dat[bad_soilvue_mask]

columns_for_drop = final_dat.columns[final_dat.columns.str.contains('EC_3')].append(
    final_dat.columns[final_dat.columns.str.contains('K_3')]).append(
        final_dat.columns[final_dat.columns.str.contains('SWC_3')]).append(
            final_dat.columns[final_dat.columns.str.contains('TS_3')])

add_date = pd.to_datetime(['2024-07-14 20:30', '2025-04-17 8:30'])
new_drop_indices = bad_data.index.append(add_date)
print(f'Number of soilvue values to drop: {len(new_drop_indices)}')
final_dat.loc[final_dat.index.isin(new_drop_indices), columns_for_drop] = np.nan


## Wind Direction

In [ ]:
# Wind Direction was off by 10 degrees in the Public table until the update date
# was 217, but actually 227
wind_update_date = pd.to_datetime('2025-12-02 09:13')
old_azimuth = 217
new_azimuth = 227
mask = final_dat.index < wind_update_date
diff = new_azimuth - old_azimuth
final_dat.loc[mask, 'WD_1_1_1'] = (final_dat.loc[mask, 'WD_1_1_1'] + diff) % 360

In [ ]:
# Young and Irgason off by about 80 degrees
wd_diff_correct = ((final_dat['WD_1_1_1'] -final_dat['WD_1_1_2'] + 180) % 360) - 180
print(wd_diff_correct.median())
wind_diff_final = -80

final_dat['WD_1_1_2'] = (final_dat.WD_1_1_2+wind_diff_final)% 360

## Misc.

In [ ]:
# Barometric Pressure
drop_date = pd.to_datetime(
    ['2023-07-20 09:00', '2023-05-25 9:00', '2025-05-31 15:30'])
mask = final_dat.index.isin(drop_date)
final_dat.loc[mask, 'PA_1_1_1'] = np.nan

In [ ]:
# SET ALB to zero if component values are zero
mask = ((final_dat.SW_IN_1_1_1.isna()) | (final_dat.SW_OUT_1_1_1.isna())) & (~final_dat.ALB_1_1_1.isna())
final_dat.loc[mask, ['SW_IN_1_1_1', 'SW_OUT_1_1_1', 'ALB_1_1_1']] = np.nan

mask = ((final_dat.SW_IN_1_1_2.isna()) | (final_dat.SW_OUT_1_1_2.isna())) & (~final_dat.ALB_1_1_2.isna())
final_dat.loc[mask, ['SW_IN_1_1_2', 'SW_OUT_1_1_2', 'ALB_1_1_2']] = np.nan

In [ ]:
# Analog Data Issue in Early Data
bad_start_date = pd.to_datetime('2023-05-27 2:30')
bad_end_date = pd.to_datetime('2023-05-29 23:30')
cols = ['NETRAD_1_1_1', 'ALB_1_1_1', 
        'LW_IN_1_1_1', 'LW_OUT_1_1_1',
        'SW_IN_1_1_1', 'SW_OUT_1_1_1',
        'G_PLATE_1_1_1', 'G_PLATE_2_1_1', 'G_1_1_1', 'G_2_1_1',
        'TS_1_1_1', 'TS_2_1_1', 'SG_1_1_1', 'SG_2_1_1']
mask = (final_dat.index>bad_start_date) & (final_dat.index<bad_end_date)
final_dat.loc[mask, cols] = np.nan

In [ ]:
# LEAF WETNESS 
# sensor #2 appears to go bad
bad_date = pd.to_datetime('2025-05-30 5:00')
mask = final_dat.index>=bad_date
vars_to_na = ['LWMV_1_2_1', 'LWMDRY_1_2_1', 'LWMCON_1_2_1','LEAF_WET_1_2_1']
final_dat.loc[mask, vars_to_na] = np.nan

In [ ]:
# various instrument footprint things

mask = final_dat.FP_DIST_INTRST>1000
final_dat.loc[mask, 'FP_DIST_INTRST'] = np.nan

mask = final_dat.UPWND_DIST_INTRST<180
final_dat.loc[mask, 'UPWND_DIST_INTRST'] = np.nan

In [ ]:
# CLEAN TEMPERATURE SPIKES

drop_dates = pd.to_datetime(['2024-04-06 06:00:00', 
               '2025-01-20 19:30:00', 
               '2025-01-20 20:00:00',
               '2025-01-20 20:30:00', 
               '2025-07-07 01:00:00',
               '2025-07-07 02:30:00'])

mask = final_dat.index.isin(drop_dates)

final_dat.loc[mask, 'TA_1_2_1'] = np.nan

## Flagging

In [ ]:
# H2O FLAGS
final_dat['H2O_SIG_FLAG_1_1_1'] = 0

low_sig_mask = final_dat.H2O_SIG_STRGTH_MIN_1_1_1<0.8
final_dat.loc[low_sig_mask, ['H2O_SIG_FLAG_1_1_1']] = 1

flag_cols = ['H2O_SIG_FLAG_1_1_1']

# continuous low signal stretches
final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2023-08-07 8:00',
                         '2023-10-26 8:30',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2024-04-22 8:30',
                         '2024-06-04 09:30',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2024-07-10 18:00',
                         '2024-08-22 07:00',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2025-04-14 07:30',
                         '2025-05-12 17:00',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2025-07-02 07:00',
                         '2025-07-16 06:30',
                         2)


In [ ]:
# CO2 FLAGS

final_dat['CO2_SIG_FLAG_1_1_1'] = 0

low_sig_mask = final_dat.CO2_SIG_STRGTH_MIN_1_1_1<0.8
final_dat.loc[low_sig_mask, ['CO2_SIG_FLAG_1_1_1']] = 1

flag_cols = ['CO2_SIG_FLAG_1_1_1']

# continuous low signal stretches
final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2023-08-07 8:00',
                         '2023-10-26 8:30',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2024-04-22 8:30',
                         '2024-06-04 09:30',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2024-07-10 18:00',
                         '2024-08-22 07:00',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2025-04-14 07:30',
                         '2025-05-12 17:00',
                         2)

final_dat = data_cleaning.apply_internal_flags(final_dat, 
                         flag_cols,
                         '2025-07-02 07:00',
                         '2025-07-16 06:30',
                         2)

In [ ]:
# Wind Flagging
wind_var = 'WD_1_1_1'
wind_flag = wind_var+'_FLAG'

final_dat[wind_flag] = 0

# set flag for moderately bad wind direction
bad_wind_flag2 = (final_dat[wind_var]<32) | (final_dat[wind_var]>212)
final_dat.loc[bad_wind_flag2, wind_flag] = 1

final_dat[wind_flag].value_counts()

# Export Data

In [ ]:
# add variables for data review
final_dat['day_of_year'] = final_dat.index.dayofyear
final_dat['time_of_day'] = final_dat.index.hour + final_dat.index.minute/60
final_dat['days_since_20240101'] = (final_dat.index - pd.Timestamp('2024-01-01')).days

In [ ]:
file_name = f"{station}_{date_range}_qc.parquet"
file_path = Path(f'{parquet_path}', "qc", f'{file_name}')
final_dat.to_parquet(file_path)
from datetime import datetime
print(f'Data exported {datetime.now()}')
print(f'Exported data to {file_path}')